# Notebook 02 — Graph-Enhanced ETA Prediction
**Run after 01_graph_bottleneck.ipynb**

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')
os.makedirs('outputs',exist_ok=True)
print('Imports done')

## 1. Load Saved Outputs

In [ ]:
df     = pd.read_csv('outputs/processed_df.csv')
hub_df = pd.read_csv('outputs/hub_metrics.csv')
with open('outputs/graph.pkl','rb') as f: G = pickle.load(f)
print(f'df: {df.shape}  |  Nodes: {G.number_of_nodes()}  |  Edges: {G.number_of_edges()}')
print(f'Columns: {df.columns.tolist()}')

## 2. Feature Engineering

In [ ]:
# Parse datetime
if df['trip_creation_time'].dtype == object:
    df['trip_creation_time'] = pd.to_datetime(df['trip_creation_time'],errors='coerce')
    df['hour']       = df['trip_creation_time'].dt.hour
    df['day_of_week']= df['trip_creation_time'].dt.dayofweek

df['route_type_enc']      = (df['route_type']=='FTL').astype(int)
df['is_cutoff_enc']       = df['is_cutoff'].astype(int)
df['segment_delay_ratio'] = (df['segment_actual_time']/df['segment_osrm_time']).clip(0.1,10)

# Map hub graph metrics
between_map = dict(zip(hub_df['hub_id'],hub_df['betweenness']))
in_deg_map  = dict(zip(hub_df['hub_id'],hub_df['in_degree']))
out_deg_map = dict(zip(hub_df['hub_id'],hub_df['out_degree']))
breach_map  = dict(zip(hub_df['hub_id'],hub_df['avg_breach_rate']))
risk_map    = dict(zip(hub_df['hub_id'],hub_df['risk_score']))
clust_map   = dict(zip(hub_df['hub_id'],hub_df['clustering']))

df['src_betweenness'] = df['source_center'].map(between_map).fillna(0)
df['src_in_degree']   = df['source_center'].map(in_deg_map).fillna(0)
df['src_out_degree']  = df['source_center'].map(out_deg_map).fillna(0)
df['src_breach_rate'] = df['source_center'].map(breach_map).fillna(0)
df['src_risk_score']  = df['source_center'].map(risk_map).fillna(0)
df['src_clustering']  = df['source_center'].map(clust_map).fillna(0)
df['dst_betweenness'] = df['destination_center'].map(between_map).fillna(0)
df['dst_in_degree']   = df['destination_center'].map(in_deg_map).fillna(0)
df['dst_breach_rate'] = df['destination_center'].map(breach_map).fillna(0)
df['dst_risk_score']  = df['destination_center'].map(risk_map).fillna(0)

print('Features added!')

## 3. Define Features & Split

In [ ]:
TARGET = 'segment_actual_time'

BASELINE_FEATURES = [
    'segment_osrm_time','segment_osrm_distance','segment_factor',
    'actual_time','osrm_time','osrm_distance','factor',
    'cutoff_factor','is_cutoff_enc',
    'hour','day_of_week','route_type_enc',
]

GRAPH_FEATURES = [
    'src_betweenness','src_in_degree','src_out_degree',
    'src_breach_rate','src_risk_score','src_clustering',
    'dst_betweenness','dst_in_degree','dst_breach_rate','dst_risk_score',
]

# Use only columns that actually exist
bline_cols = [f for f in BASELINE_FEATURES if f in df.columns]
graph_cols = [f for f in GRAPH_FEATURES     if f in df.columns]
all_cols   = bline_cols + graph_cols

print(f'Baseline features: {len(bline_cols)} → {bline_cols}')
print(f'Graph features:    {len(graph_cols)} → {graph_cols}')

df_model = df[all_cols+[TARGET]].dropna()
X = df_model[all_cols]
y = df_model[TARGET]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 4. Model A — Baseline (No Graph Features)

In [ ]:
baseline = XGBRegressor(n_estimators=300,learning_rate=0.05,max_depth=6,
                         subsample=0.8,random_state=42,n_jobs=-1)
baseline.fit(X_train[bline_cols],y_train,
             eval_set=[(X_test[bline_cols],y_test)],verbose=50)

preds_base    = baseline.predict(X_test[bline_cols])
mae_base      = mean_absolute_error(y_test,preds_base)
rmse_base     = np.sqrt(mean_squared_error(y_test,preds_base))
r2_base       = r2_score(y_test,preds_base)
within15_base = np.mean(np.abs(preds_base-y_test)/y_test.clip(lower=0.01)<0.15)*100

print(f'Baseline MAE:       {mae_base:.2f}')
print(f'Baseline RMSE:      {rmse_base:.2f}')
print(f'Baseline R²:        {r2_base:.4f}')
print(f'Baseline within 15%:{within15_base:.1f}%')

## 5. node2vec Embeddings

In [ ]:
NODE2VEC_AVAILABLE = False
try:
    from node2vec import Node2Vec
    print('Training node2vec... (~2-3 min)')
    n2v_model = Node2Vec(G,dimensions=32,walk_length=20,num_walks=80,workers=4,quiet=True)
    n2v       = n2v_model.fit(window=10,min_count=1,batch_words=4)
    n2v.save('outputs/node2vec_model.pkl')
    def get_emb(fid):
        try:    return n2v.wv[str(fid)]
        except: return np.zeros(32)
    NODE2VEC_AVAILABLE = True
    print('node2vec ready!')
except ImportError:
    print('node2vec not installed → pip install node2vec')
    print('Using graph metric features only for enhanced model')

## 6. Model B — Graph-Enhanced

In [ ]:
if NODE2VEC_AVAILABLE:
    src_emb = np.stack(df_model['source_center'].apply(get_emb).values)
    dst_emb = np.stack(df_model['destination_center'].apply(get_emb).values)
    src_df  = pd.DataFrame(src_emb,columns=[f'src_emb_{i}' for i in range(32)],index=df_model.index)
    dst_df  = pd.DataFrame(dst_emb,columns=[f'dst_emb_{i}' for i in range(32)],index=df_model.index)
    X_graph = pd.concat([X,src_df,dst_df],axis=1)
    print(f'Graph-enhanced features: {X_graph.shape[1]} (baseline {len(bline_cols)} + graph {len(graph_cols)} + embeddings 64)')
else:
    X_graph = X.copy()
    print(f'Graph metrics only: {X_graph.shape[1]} features')

Xg_train = X_graph.loc[X_train.index]
Xg_test  = X_graph.loc[X_test.index]

graph_model = XGBRegressor(n_estimators=300,learning_rate=0.05,max_depth=6,
                            subsample=0.8,random_state=42,n_jobs=-1)
graph_model.fit(Xg_train,y_train,eval_set=[(Xg_test,y_test)],verbose=50)

preds_graph    = graph_model.predict(Xg_test)
mae_graph      = mean_absolute_error(y_test,preds_graph)
rmse_graph     = np.sqrt(mean_squared_error(y_test,preds_graph))
r2_graph       = r2_score(y_test,preds_graph)
within15_graph = np.mean(np.abs(preds_graph-y_test)/y_test.clip(lower=0.01)<0.15)*100

print(f'Graph MAE:       {mae_graph:.2f}')
print(f'Graph RMSE:      {rmse_graph:.2f}')
print(f'Graph R²:        {r2_graph:.4f}')
print(f'Graph within 15%:{within15_graph:.1f}%')

## 7. Benchmark Table — The Graph Advantage

In [ ]:
osrm_preds   = df_model['segment_osrm_time']
osrm_mae     = mean_absolute_error(df_model[TARGET], osrm_preds)
osrm_within  = np.mean(np.abs(osrm_preds-df_model[TARGET])/df_model[TARGET].clip(lower=0.01)<0.15)*100
osrm_r2      = r2_score(df_model[TARGET], osrm_preds)

results = pd.DataFrame({
    'Model':        ['OSRM (current system)','Baseline XGBoost','Graph-Enhanced XGBoost'],
    'MAE':          [round(osrm_mae,2),  round(mae_base,2),   round(mae_graph,2)],
    'RMSE':         [round(np.sqrt(mean_squared_error(df_model[TARGET],osrm_preds)),2), round(rmse_base,2), round(rmse_graph,2)],
    'R2':           [round(osrm_r2,4),   round(r2_base,4),    round(r2_graph,4)],
    'Within_15pct': [round(osrm_within,1),round(within15_base,1),round(within15_graph,1)],
})

print('='*65)
print('  THE GRAPH ADVANTAGE — BENCHMARK')
print('='*65)
print(results.to_string(index=False))
print('='*65)
print(f'MAE improvement over baseline:     {mae_base-mae_graph:.2f}')
print(f'Within-15% improvement:            {within15_graph-within15_base:.1f}%')

results.to_csv('outputs/model_benchmark.csv',index=False)

## 8. Visualize Results

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,5))
colors = ['#4a4a6a','#EF9F27','#1D9E75']
models = results['Model'].tolist()

axes[0].bar(models,results['MAE'],color=colors,edgecolor='white')
axes[0].set_title('MAE (lower=better)',fontweight='bold')
axes[0].tick_params(axis='x',rotation=15)
for i,v in enumerate(results['MAE']):
    axes[0].text(i,v+0.01,str(v),ha='center',fontweight='bold')

axes[1].bar(models,results['Within_15pct'],color=colors,edgecolor='white')
axes[1].set_title('% Within 15% of Actual (higher=better)',fontweight='bold')
axes[1].tick_params(axis='x',rotation=15)
for i,v in enumerate(results['Within_15pct']):
    axes[1].text(i,v+0.3,f'{v}%',ha='center',fontweight='bold')

sample = np.random.choice(len(y_test),min(500,len(y_test)),replace=False)
axes[2].scatter(y_test.iloc[sample],preds_graph[sample],alpha=0.4,color='#1D9E75',s=20)
lims = [min(y_test.min(),preds_graph.min()),max(y_test.max(),preds_graph.max())]
axes[2].plot(lims,lims,'r--',lw=2,label='Perfect')
axes[2].set_xlabel('Actual Time')
axes[2].set_ylabel('Predicted Time')
axes[2].set_title('Actual vs Predicted\n(Graph Model)',fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.savefig('outputs/model_benchmark.png',dpi=150,bbox_inches='tight')
plt.show()

## 9. Feature Importance

In [ ]:
fi = pd.DataFrame({'feature':Xg_train.columns,'importance':graph_model.feature_importances_
}).sort_values('importance',ascending=False).head(20)

fig,ax = plt.subplots(figsize=(10,7))
ax.barh(fi['feature'],fi['importance'],color='#1D9E75',edgecolor='white')
ax.set_title('Top 20 Feature Importances — Graph Model',fontweight='bold')
ax.set_xlabel('Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('outputs/feature_importance.png',dpi=150,bbox_inches='tight')
plt.show()
print('Top 10:')
print(fi.head(10).to_string(index=False))

## 10. Save Models

In [ ]:
with open('outputs/baseline_model.pkl','wb') as f: pickle.dump(baseline,f)
with open('outputs/graph_model.pkl','wb') as f:    pickle.dump(graph_model,f)
with open('outputs/graph_features.pkl','wb') as f: pickle.dump(list(Xg_train.columns),f)
print('Models saved!')